# Experiment 13 — DINOv2 ViT-L/14 Encoder — Colab Run

A different hypothesis from exp12: instead of task-adjacent transfer (Task 2's own segmentation-trained encoder), this tries a much larger, general-purpose, self-supervised backbone -- DINOv2 ViT-L/14 (303M params, timm `vit_large_patch14_dinov2.lvd142m`, pretrained on LVD-142M, zero wound/medical images). Two modes: `frozen` (linear probe, only CornHead trains, cheap) and `finetune` (all 303M params train, much smaller lr/batch -- see README's "Fine-tune güvenliği"). Downloads ~1.2GB of pretrained weights on first run -- GPU runtime required.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the PIID staging dataset

Same `wound-stage-classification.zip` task_3's own notebook uses.

In [ ]:
import glob

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-stage-classification"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if not os.path.exists(DATASET_DIR):
    os.symlink(candidates[0], DATASET_DIR)
print(DATASET_DIR, "->", os.path.realpath(DATASET_DIR))

## 4. Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/more_experiments/exp/exp13_dinov2_backbone/requirements.txt

## 5. Architecture sanity check (no weight download yet)

`pretrained=False` -- just shapes/freeze-logic, same checks as local.

In [ ]:
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp13_dinov2_backbone/scripts")
!python test_model.py

## 6. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp13_dinov2_backbone_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/exp13_results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 7. Train — frozen mode (linear probe), all 3 folds

Cheap: only ~3K params train, 100 epochs. Downloads DINOv2 ViT-L's real pretrained weights (~1.2GB) on the first fold.

In [ ]:
!python train.py --mode frozen --workers 4

## 8. (Optional) Train — finetune mode, all 3 folds

Expensive: all 303M params train, much smaller lr/batch, 30 epochs. Run step 7 first and look at the frozen-mode numbers before deciding whether this is worth the compute.

In [ ]:
!python train.py --mode finetune --workers 4

## 9. Compare against the ImageNet-init references

`mit_b1__corn` / `mit_b2__corn` read directly from task_3's OWN `ablation_results.csv` (not retrained, not modified).

In [ ]:
!python compare_results.py